# 06 Case study: Bhote Koshi / Trishuli flash flood, 26 Aug 2026 (EMSR927)

Runs the full pipeline for the event and compares it with Copernicus EMS EMSR927. **EMSR927 is evaluation only**: it is read by
`src.validate` and by nothing else, and no threshold was changed after looking at it.

Prerequisites (see README): CDSE credentials in `.env`, SNAP `gpt` on the PATH, and the EMSR927 vector download in `data/emsr927/`.
Cells that need a missing prerequisite print what is missing and carry on, so the notebook never shows invented numbers.
If `FLOOD_DEMO=1` is set it runs on the synthetic demo town instead, and every figure is labelled synthetic.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt

AOI, DATE = "config/aoi_trishuli.yaml", "2026-08-26"
EMSR = Path("data/emsr927")
DEMO = os.getenv("FLOOD_DEMO") == "1"
if DEMO:
    from src.demo_data import build
    d = build(Path("data/demo")); OUT, OSM = d["outputs"], d["osm"]; WD = Path("data/demo")
    for mod in ("damage", "cutoff"):   # the tail stages read the synthetic masks
        subprocess.run([sys.executable, "-m", f"src.{mod}", "--config", AOI, "--outputs", str(OUT), "--osm", str(OSM),
                        "--flood-source", "baseline"], check=True)
else:
    WD = Path(f"data/runs/Trishuli_2026_{DATE}"); OUT, OSM = WD / "outputs", WD / "raw/osm/osm_pre_event.gpkg"
print("outputs:", OUT, "| synthetic:", DEMO)

## 1. Run the pipeline (skip if already done)
Set `RUN = True` to execute ingest, preprocessing, flood map, damage, cut-off and the PDF.

In [ ]:
RUN = False
if RUN and not DEMO:
    r = subprocess.run([sys.executable, "-m", "src.pipeline", "--aoi", AOI, "--date", DATE], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
log = WD / "pipeline_log.json"
print(log.read_text()[:1500] if log.exists() else "no pipeline_log.json yet: run the pipeline first")

## 2. Validate against EMSR927
Strict and tolerant (±1 px = 10 m) scores inside the EMSR areas of interest only. Run `python -m src.validate inspect --emsr-dir data/emsr927` once to check file and column names.

In [ ]:
from src import validate
rep = Path("reports"); val = None
if not (OUT / "flood_mask.tif").exists():
    print("MISSING: flood masks in", OUT, "- run section 1 first")
elif not EMSR.exists() and not DEMO:
    print("MISSING:", EMSR, "- download the EMSR927 vectors (observedEvent, areaOfInterest, builtUp, transportation)")
elif DEMO:
    print("demo mode: EMSR comparison skipped (no EMSR files exist for the synthetic town; see tests/test_day6.py)")
else:
    val = validate.run(AOI, EMSR, OUT, OSM, rep, emsr_image_date=None)  # set emsr_image_date='YYYY-MM-DD' to record the timing gap
    print(open(rep / "validation.md").read())

## 3. Side-by-side maps and confusion matrix

In [ ]:
try:
    from IPython.display import Image, display
except ImportError:
    display = Image = print
figs = sorted(Path("reports").glob("fig_val_*.png")) if val else []
for f in figs: display(Image(str(f)))
print("no validation figures yet" if not figs else "")

## 4. Flood mask: scores, and where they come from
The tolerant score forgives a one-pixel misregistration, which matters because EMSR is drawn on imagery of about 1 m and we work at 10 m.
The gap between strict and tolerant scores is the part attributable to resolution rather than to missed or false water.

In [ ]:
if val:
    f = val["flood"]["flood_or_debris"]
    rows = [("strict", f["strict"]["iou"], f["strict"]["f1"], f["strict"]["precision"], f["strict"]["recall"]),
            ("tolerant ±%d px" % f["tolerant"]["tol_px"], None, f["tolerant"]["f1"], f["tolerant"]["precision"], f["tolerant"]["recall"])]
    fmt = lambda x: "n/a" if x is None else f"{x:.3f}"
    print(f"{'':16s}{'IoU':>8s}{'F1':>8s}{'Prec':>8s}{'Rec':>8s}")
    for r in rows: print(f"{r[0]:16s}" + "".join(f"{fmt(x):>8s}" for x in r[1:]))
    print("flood water only (no debris), strict IoU:", fmt(val['flood']['flood_only']['strict']['iou']))
    print("per EMSR area:", [(a['aoi'], fmt(a['iou'])) for a in val['flood']['per_aoi']])

## 5. Buildings, roads, bridges, cut-off settlements

In [ ]:
if val:
    for k in ("buildings", "roads", "bridges"):
        print(k, json.dumps(val.get(k), indent=1))
    c = val.get("cutoff", {})
    for lab in ("cut_off", "cut_off_or_if_bridges_fail"):
        if lab in c: print(lab, {k: c[lab][k] for k in ("ours", "emsr_derived", "both", "jaccard", "only_ours", "only_emsr_derived")})
    print(c.get("note", ""))

## 6. Honest reading of the disagreements
Fill this in after looking at the maps. Do not explain a disagreement with a cause you have not checked.

| Observation | Candidate cause | How to check |
|---|---|---|
| Our extent is narrower than EMSR along the river | 10 m pixels, a narrow channel | tolerant recall vs strict recall |
| Our extent is wider | acquisitions differ in time (flood receded in the EMSR image or grew in ours) | `days_between_s1_post_and_emsr_image` in `validation.json` |
| Gaps in the gorge | radar shadow/layover | `s1_unobservable_fraction` in `flood_report.json` |
| Missing flood near the confluence | cloud on Sentinel-2, so radar only | `s2_clear_fraction_both_dates` |
| Buildings flagged but "no visible damage" in EMSR | exposure is not damage | precision is a lower bound |
| EMSR buildings with no OSM footprint | OSM incomplete | `osm_match_rate_affected` |
| Flood on a glacier or high snowfield | wet snow | corridor filter, relative-elevation layer |

In [ ]:
if (OUT / "flood_report.json").exists():
    print(json.dumps(json.load(open(OUT / "flood_report.json")), indent=1)[:1500])